In [ ]:
import os
import requests
import json
import torch

from pathlib import Path
from datasets import load_dataset



In [ ]:

import sys
sys.path.append("..")


from src.tokenizer import ScratchyTokenizer

In [ ]:
data_path = Path("../data")
ckpt_path = Path("../ckpt")

ckpt_path.mkdir(exist_ok=True, parents=True)

warandpeace_path = data_path / "warandpeace.txt"


tokenizer_path = ckpt_path / 'tokenizer'
tokenizer_path.mkdir(exist_ok=True, parents=True)

In [ ]:
with open(warandpeace_path, 'r') as file:
    warandpeace = file.read()

In [ ]:
tokenizer_scratchy_path = tokenizer_path / "scratchy.json"

with open(tokenizer_scratchy_path, 'r') as file:
    chars = json.load(file)

In [ ]:

tokenizer = ScratchyTokenizer(chars)
data = torch.tensor(tokenizer.encode(warandpeace), dtype=torch.long)
data.shape

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

In [ ]:

torch.manual_seed(1337)

In [ ]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
class BigramLM(torch.nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)

            logits = logits[:, -1, :] # (B, C)

            probs = F.softmax(logits, dim=-1) # (B, C)

            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            idx = torch.cat([idx, idx_next], dim=1) # (B, C + 1)
        return idx
    

In [ ]:

model = BigramLM(tokenizer.vocab_size)
m = model.to(device)

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [ ]:

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step: {iter}: train_loss {losses["train"]:.4f}, val_loss {losses["val"]:.4f}")

    inputs, targets = get_batch("train")

    logits, loss = model(inputs, targets)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


In [ ]:

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(tokenizer.decode(m.generate(context, max_new_tokens=500)[0].tolist()))

In [ ]:
torch.manual_seed(1337)

B, T, C = 4, 8, 2 # batch, time, channels
x = torch.randn(B, T, C)

x.shape

In [ ]:
# x[b, t] = mean_{i < t} x[b,i]

xbow = torch.zeros(B, T, C)
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1] # (t, C)
        xbow[b, t] = torch.mean(xprev, dim=0) 

xbow[0]

In [ ]:
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b

print(a)
print(b)
print(c)

In [ ]:
weights = torch.tril(torch.ones(T, T))
weights = weights / weights.sum(1, keepdim=True)
xbow = weights @ x # (T, T) @ (B, T, C) -> (B, T, C)
xbow[0]

In [ ]:
tril = torch.tril(torch.ones(T, T))
weights = torch.zeros(T, T)
weights = weights.masked_fill(tril == 0, float("-inf"))
weights = F.softmax(weights, dim=-1)
xbow3 = weights @ x
xbow3[0]

In [ ]:
# self attention

torch.manual_seed(1337)

B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B, T, head_size)
q = query(x) # (B, T, head_size)

weights = q @ k.transpose(-2, -1) * head_size ** -0.5# (B, T, head_size) @ (B, head_size, T) -> (B, T, T)

tril = torch.tril(torch.ones(T, T))
weights = weights.masked_fill(tril == 0, float("-inf"))
weights = F.softmax(weights, dim=-1)

v = value(x)
out = weights @ v

out.shape

In [ ]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

w = q @ k.transpose(-2, -1) * head_size ** -.5




In [ ]:
k.var()

In [ ]:
q.var()

In [ ]:
w.var()

In [ ]:
weights[0]

In [ ]:
print(k.shape)

print(k.transpose(-2, -1).shape)



In [ ]:
# scaled dot product attention

class Head(nn.Module):
    def __init__(self, n_embedding, context_size, head_size):
        super().__init__()
        self.n_embedding = n_embedding
        self.context_size = context_size
        self.head_size = head_size
        self.query = nn.Linear(self.n_embedding, self.head_size, bias=False)
        self.key = nn.Linear(self.n_embedding, head_size, bias=False)
        self.value = nn.Linear(self.n_embedding, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(self.context_size, self.context_size)))

    def forward(self, x):
        B, T, C = x.shape

        q = self.query(x)  # (B, T, C)
        k = self.key(x)  # (B, T, C)

        w = k @ q.transpose(-2, -1) * self.head_size ** -0.5  # (B, T, C) @ (B, C, T) -> (B, T, T)
        w = w.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # (B, T, T)
        w = F.softmax(w, dim=-1)

        v = self.value(x) # (B, T, C)
        y = w @ v  # (B, T, T) @ (B, T, C) -> (B, T, C)
        return y

In [ ]:
class AttentionLM(torch.nn.Module):

    def __init__(self, head_size, n_embedding, context_size, vocab_size):
        super().__init__()
        self.n_embedding = n_embedding
        self.context_size = context_size
        self.head_size = head_size
        self.vocab_size = vocab_size
        
        self.token_embedding_table = nn.Embedding(vocab_size, n_embedding)
        self.position_embedding_table = nn.Embedding(context_size, n_embedding)

        self.sa_head = Head(n_embedding, context_size, n_embedding)
        self.lm_head = nn.Linear(n_embedding, vocab_size)

        
    def forward(self, idx, targets=None):
        B, T = idx.shape
        device = idx.device

        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)

        x = tok_emb + pos_emb # (B, T, C)
        x = self.sa_head(x) # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_size:]
            logits, loss = self(idx_cond)

            logits = logits[:, -1, :] # (B, C)

            probs = F.softmax(logits, dim=-1) # (B, C)

            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            idx = torch.cat([idx, idx_next], dim=1) # (B, C + 1)
        return idx
    

In [ ]:
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embeddings = 32
head_size = 16


model = AttentionLM(head_size, n_embeddings, block_size, tokenizer.vocab_size)
m = model.to(device)

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:

learning_rate = 1e-4

In [ ]:
for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step: {iter}: train_loss {losses["train"]:.4f}, val_loss {losses["val"]:.4f}")

    inputs, targets = get_batch("train")

    logits, loss = model(inputs, targets)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


In [ ]:

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(tokenizer.decode(m.generate(context, max_new_tokens=500)[0].tolist()))